<a href="https://colab.research.google.com/github/huseyinTozluyurt/Flyrank-Internship-MachineLearning/blob/main/work/notebooks/milestone_2_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Missingness and leakage-safe preprocessing

This stage follows the completed CSV audit and uses `data/raw/content_refresh_anonymized.csv`. It compares preprocessing choices for the existing current-label task, using held-out clients. It does not connect to DuckDB and does not claim future-warning performance.

Audit facts used here: 30,000 unique pages across 32 clients; the target is `trend_direction == 'down'`; `avg_position == 0` means no position data; `word_count` and `char_count` have substantial, content-type-linked missingness. Those sparse text fields are not in the current agreed feature set, so they will not drive imputation for this model.

In [1]:
# Open from your FlyRank repository checkout, or clone it in Colab.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/huseyinTozluyurt/Flyrank-Internship-MachineLearning'
REPO_DIR = 'Flyrank-Internship-MachineLearning'
CSV_PATH = Path('data/raw/content_refresh_anonymized.csv')
if IN_COLAB and not CSV_PATH.exists():
    if not Path(REPO_DIR).is_dir():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    CSV_PATH = Path('data/raw/content_refresh_anonymized.csv')
else:
    while not CSV_PATH.exists() and Path.cwd() != Path('/'):
        os.chdir('..')
        CSV_PATH = Path('data/raw/content_refresh_anonymized.csv')
if not CSV_PATH.exists():
    raise FileNotFoundError('Expected data/raw/content_refresh_anonymized.csv in the FlyRank repo.')
print('CSV:', CSV_PATH.resolve())

CSV: /content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv


In [2]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline

df = pd.read_csv(CSV_PATH)
required = ['content_id','client_id','trend_direction','days_since_last_update',
            'content_age_days','impressions_90d','avg_position','ctr']
missing = [c for c in required if c not in df.columns]
if missing: raise ValueError(f'Missing required fields: {missing}')
df['trend_direction'] = df['trend_direction'].astype('string').str.strip().str.lower()
df['is_declining_label'] = df['trend_direction'].map(
    {'down':1,'new':0,'up':0,'flat':0,'stable':0}
).astype('Int64')

# A page with avg_position == 0 has no observed position. Keep it missing.
FEATURES = ['days_since_last_update','content_age_days','impressions_90d','avg_position','ctr']
X = df[FEATURES].apply(pd.to_numeric, errors='coerce').copy()
X['avg_position'] = X['avg_position'].mask(X['avg_position'].eq(0))
X['staleness_pos_interaction'] = X['days_since_last_update'] * X['avg_position']
FEATURES = list(X.columns)

valid = (df['is_declining_label'].notna() & df['content_id'].notna() &
         df['client_id'].notna() & ~df['content_id'].duplicated(keep=False))
X = X.loc[valid].reset_index(drop=True)
y = df.loc[valid,'is_declining_label'].astype(int).reset_index(drop=True)
groups = df.loc[valid,'client_id'].reset_index(drop=True)
print(f'Eligible modelling rows: {len(X):,}; clients: {groups.nunique():,}; decline rate: {y.mean():.3f}')
print('Raw-feature missingness:')
display(X.isna().mean().sort_values(ascending=False).rename('missing_rate').to_frame())

Eligible modelling rows: 30,000; clients: 32; decline rate: 0.542
Raw-feature missingness:


,missing_rate
staleness_pos_interaction,0.040167
avg_position,0.040167
content_age_days,0.000000
days_since_last_update,0.000000
impressions_90d,0.000000
ctr,0.000000


## Strategy comparison

We compare three treatments using the same `RandomForestClassifier` and the same client-held-out splits:

1. **Complete-case reference:** train and evaluate only rows with all contracted features observed. This reveals the cost of dropping incomplete pages.
2. **Median imputation:** fit medians on each training fold, then apply them to that fold's holdout.
3. **Median + missingness indicators:** same train-only median imputation, with explicit indicators for fields missing in each row.

The imputer is inside each model pipeline so validation data never influences medians. We use 5 `GroupShuffleSplit` splits by `client_id`; the small number of clients makes results exploratory, so report the spread as well as the mean. No imputation is written back into the source CSV.

In [3]:
RANDOM_STATE = 42
N_SPLITS = 5
TEST_SIZE = 0.25
K = 50
gss = GroupShuffleSplit(n_splits=N_SPLITS, test_size=TEST_SIZE, random_state=RANDOM_STATE)

def precision_at_k(scores, labels, k=K):
    scores, labels = np.asarray(scores), np.asarray(labels)
    n = min(k, len(labels))
    if n == 0: return np.nan
    order = np.argsort(-scores)[:n]
    return float(labels[order].mean())

def score_split(y_true, probabilities):
    out = {'average_precision': float(average_precision_score(y_true, probabilities)),
           'precision_at_k': precision_at_k(probabilities, y_true),
           'n_test': int(len(y_true))}
    out['roc_auc'] = float(roc_auc_score(y_true, probabilities)) if pd.Series(y_true).nunique() == 2 else np.nan
    return out

def model_pipeline():
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_leaf=2,
                                         class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1))
    ])

rows=[]
for split_id,(train_idx,test_idx) in enumerate(gss.split(X,y,groups),start=1):
    train_clients=set(groups.iloc[train_idx]); test_clients=set(groups.iloc[test_idx])
    assert train_clients.isdisjoint(test_clients), 'Client leakage across split.'
    X_train,X_test=X.iloc[train_idx].copy(),X.iloc[test_idx].copy()
    y_train,y_test=y.iloc[train_idx].copy(),y.iloc[test_idx].copy()

    # Complete-case reference: impute is unnecessary after restricting to complete rows.
    train_cc=X_train.notna().all(axis=1); test_cc=X_test.notna().all(axis=1)
    if train_cc.sum() and test_cc.sum() and y_train[train_cc].nunique()==2:
        cc=RandomForestClassifier(n_estimators=200,max_depth=5,min_samples_leaf=2,
                                  class_weight='balanced',random_state=RANDOM_STATE,n_jobs=-1)
        cc.fit(X_train.loc[train_cc],y_train.loc[train_cc])
        probs=cc.predict_proba(X_test.loc[test_cc])[:,1]
        rows.append({'split':split_id,'strategy':'complete_case',**score_split(y_test.loc[test_cc],probs)})

    # Median-only; train-fold medians are learned by the pipeline.
    med=model_pipeline(); med.fit(X_train,y_train)
    rows.append({'split':split_id,'strategy':'median',**score_split(y_test,med.predict_proba(X_test)[:,1])})

    # Explicit missingness flags are constructed from raw values, before imputation.
    X_train_flag=X_train.copy(); X_test_flag=X_test.copy()
    for col in FEATURES:
        flag=f'{col}__missing'
        X_train_flag[flag]=X_train[col].isna().astype('int8')
        X_test_flag[flag]=X_test[col].isna().astype('int8')
    med_flags=model_pipeline(); med_flags.fit(X_train_flag,y_train)
    rows.append({'split':split_id,'strategy':'median_plus_missing_flags',
                 **score_split(y_test,med_flags.predict_proba(X_test_flag)[:,1])})

split_results=pd.DataFrame(rows)
summary=(split_results.groupby('strategy')
         .agg(splits=('split','nunique'),mean_average_precision=('average_precision','mean'),
              sd_average_precision=('average_precision','std'),mean_precision_at_50=('precision_at_k','mean'),
              sd_precision_at_50=('precision_at_k','std'),mean_roc_auc=('roc_auc','mean'),
              mean_test_rows=('n_test','mean'))
         .sort_values('mean_average_precision',ascending=False))
display(summary.round(3))

,splits,mean_average_precision,sd_average_precision,mean_precision_at_50,sd_precision_at_50,mean_roc_auc,mean_test_rows
strategy,,,,,,,
complete_case,5,0.612,0.077,0.616,0.200,0.641,5666.0
median,5,0.611,0.077,0.632,0.209,0.665,5878.8
median_plus_missing_flags,5,0.610,0.075,0.652,0.139,0.664,5878.8


## Decision and handoff

Choose the preprocessing strategy using held-out-client average precision and Precision@50, considering the split-to-split variation and complete-case row loss. Keep the simplest strategy whose performance is stable. Do not select based on a single favorable split.

A zero `avg_position` is treated as unavailable, not rank zero. Median values are fit separately inside each training fold; missingness flags preserve the data-availability signal. Word/character counts are excluded from this capstone feature set for now, so their 25.7% missingness does not force row deletion or affect this model.

The current snapshot still lacks verified feature timestamps. Any scores here describe discrimination of the supplied current trend label only. The next stage can use the selected preprocessing pipeline to compare the heuristic baseline and classifiers. The final full-warehouse pass remains necessary to validate dated windows.

In [4]:
OUTPUT_DIR=Path('work/outputs/milestone_2_preprocessing')
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
split_results.to_csv(OUTPUT_DIR/'split_strategy_results.csv',index=False)
summary.to_csv(OUTPUT_DIR/'preprocessing_strategy_summary.csv')
missing_report=X.isna().agg(['sum','mean']).T.rename(columns={'sum':'missing_count','mean':'missing_rate'})
missing_report.to_csv(OUTPUT_DIR/'model_feature_missingness.csv')
contract={
 'target':'is_declining_label derived from trend_direction == down',
 'unit':'one unique content_id within client_id',
 'features':FEATURES,
 'split':'5 repeated 75/25 GroupShuffleSplit partitions by client_id',
 'strategies':['complete_case_reference','train_fold_median','train_fold_median_plus_missing_flags'],
 'position_zero':'converted to missing before split',
 'scope':'current-label prototype; feature timestamps unverified; no early-warning claim',
}
(OUTPUT_DIR/'preprocessing_contract.json').write_text(json.dumps(contract,indent=2)+'\n',encoding='utf-8')
print('Saved preprocessing comparison to',OUTPUT_DIR.resolve())

Saved preprocessing comparison to /content/flyrank-ml-internship-starter/work/outputs/milestone_2_preprocessing
